# Train on the two ear-pressure answers

This isolated pilot starts a fresh Llama 3 8B QLoRA adapter and trains on the General and Clinician answers kept by the project owner. One optimizer update uses both answers. The four bundled validation examples check the software on other topics. This small experiment cannot establish medical answer quality.

In Colab, select a GPU runtime, enable notebook access to the Hugging Face read secret `HF_TOKEN`, and run all cells. Each run saves in a new Google Drive folder. No file upload is required.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
SOURCE_COMMIT = '189285c8b31a102b61038834f7f00928102de5cc'
REPO = Path('/content') / ('healthbrowse-pilot-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--filter=blob:none', '--branch', 'research/healthbrowse-medical-model',
                    'https://github.com/abby2395/medical_chatbot.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('The pilot checkout does not match the pinned source commit.')
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Pinned pilot source:', head)


In [ ]:
# Keep the text-model stack independent of packages preinstalled in the Colab kernel.
import venv
ENV_DIR = Path('/content') / ('healthbrowse-train-' + sys.implementation.cache_tag)
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
LAB_PYTHON = str(ENV_DIR / 'bin/python')
lab_env = os.environ.copy()
lab_env.pop('PYTHONPATH', None)
lab_env['PYTHONNOUSERSITE'] = '1'

def run_lab(*args):
    subprocess.run([LAB_PYTHON, *map(str, args)], cwd=LAB, env=lab_env, check=True)

# Target the venv directly; Colab's system Python may not ship ensurepip.
subprocess.run([sys.executable, '-m', 'pip', '--python', LAB_PYTHON,
                'install', '-e', '.[train,test]'], cwd=LAB, env=lab_env, check=True)
run_lab('-c', """import torch, transformers, accelerate, bitsandbytes
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.integrations import get_reporting_integration_callbacks
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Trainer imports passed.', 'torch', torch.__version__, 'transformers', transformers.__version__)
""")


In [ ]:
from google.colab import userdata
run_lab('-m', 'hb_model_lab.cli', 'preflight')
run_lab('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before training.'")
try:
    lab_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Enable notebook access to HF_TOKEN in Colab Secrets.") from exc


In [ ]:
run_lab('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## Prepare the two training records

The input text, exact quotes and audience-specific answers come from the pinned review file. The copy used for this experiment retains its review history and is marked illustrative. The lab’s one-step mode keeps the resulting adapter restricted to research use.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/HealthBrowseModelLab') / ('ear-pressure-' + uuid.uuid4().hex[:12])
DATA = RUN_ROOT / 'data'
DATA.mkdir(parents=True, exist_ok=False)
TRAIN = DATA / 'train.jsonl'
VALIDATION = DATA / 'validation.jsonl'
source = LAB / 'data/review/middle-ear-pressure.jsonl'
rows = [json.loads(line) for line in source.read_text().splitlines() if line.strip()]
if len(rows) != 2 or {r['input']['role'] for r in rows} != {'general', 'clinician'}:
    raise ValueError('Expected one General and one Clinician training record.')
for row in rows:
    if row['review'].get('wording_decision') != 'keep_as_written':
        raise ValueError('Expected the accepted answer wording.')
    row['example_only'] = True
TRAIN.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows))
VALIDATION.write_bytes((LAB / 'data/examples/validation.jsonl').read_bytes())
config = json.loads((LAB / 'configs/llama3-8b-qlora.json').read_text())
config.update(batch_size=1, gradient_accumulation_steps=2)
CONFIG = RUN_ROOT / 'pilot-config.json'
CONFIG.write_text(json.dumps(config, indent=2))
run_lab('-m', 'hb_model_lab.cli', 'validate', '--examples', '--train', TRAIN, '--validation', VALIDATION)
print('Run folder:', RUN_ROOT)


In [ ]:
run_lab('-m', 'hb_model_lab.cli', 'train', '--config', CONFIG, '--train', TRAIN,
        '--validation', VALIDATION, '--output', RUN_ROOT / 'model', '--smoke')
manifest = json.loads((RUN_ROOT / 'model/run-manifest.json').read_text())
adapter = RUN_ROOT / 'model/adapter/adapter_model.safetensors'
assert manifest['status'] == 'trained' and manifest['train_examples'] == 2 and adapter.is_file()
print('Pilot complete: 2 training examples.')
print('Adapter:', adapter)


The saved model is a one-step experimental adapter. The General and Clinician examples teach one topic in chat mode. The validation loss on four examples does not measure six-deck search performance or clinical accuracy.
